# CPM Scheduler — Performance Benchmark Analysis

This notebook characterises the wall-clock scaling behaviour of
`Pert.calculateScheduleWithResources()` across four dimensions:

| Dimension | Values |
|---|---|
| **Topologies** | `serial`, `fan`, `pipeline` (scheduler) · `plant_outage` (plant-scale) |
| **Pool modes** | `unconstrained`, `tight` (scheduler) · `plant`, `tight` (plant-scale) |
| **SGS variants** | `first` · `max_use_res_ranked` · `max_use_res_shuffled` · `md_knapsack` · `look_ahead` |
| **Problem sizes** | 100 – 1 500 work activities (scheduler) / 1 000 – 15 000 (plant-scale) |

Results are persisted as JSON files so the benchmarks can be reloaded without
re-running. Each `(topology, pool, SGS, n)` combination is a separate result row.

---

## Topology Descriptions

The **topology** controls how work activities are connected in the precedence graph.
Each topology is a synthetic stress-test designed to expose a specific scheduling
pattern:

| Topology | Description |
|---|---|
| `serial` | Activities form a single chain: each task must finish before the next begins. The scheduler processes one activity at a time (`_ready` set size = 1 throughout), so resource conflicts are minimal and scheduling is O(n). This is the best-case topology — it sets the CPM-limited lower bound on scheduling time. |
| `fan` | All work activities are independent and can run in parallel (no precedence edges between them). The scheduler sees the entire workload in `_ready` at once, making candidate selection maximally expensive. Under a tight resource pool this forces heavy serialisation; under unconstrained resources all activities complete simultaneously. |
| `pipeline` | Activities are grouped into fixed-size clusters (default: 10 activities per cluster). Within each cluster activities run in parallel; clusters are sequenced by a START gate → work → END gate pattern. This mimics a modular outage workflow (e.g. isolation → maintenance → restoration per system). `_ready` is bounded by the cluster size regardless of n. |
| `plant_outage` | A realistic nuclear outage topology: 16 parallel work-package pipelines, each a `pipeline`-style sequence of clusters. Approximately 15 % of pipelines carry a cross-stream precedence gate requiring the preceding pipeline to complete a milestone before the next one starts (system-isolation prerequisites). This is the production-representative topology. |

---

## Pool Descriptions

The **pool** controls how many resources of each skill type are available.
It determines how much resource contention the scheduler must resolve:

| Pool | Description |
|---|---|
| `unconstrained` | Each activity is assigned one resource unit from an unlimited pool (capacity = n). No resource conflicts ever arise; the scheduler only needs to respect precedence edges. Timing in this mode isolates the graph-traversal cost of the scheduler, independent of contention resolution. |
| `tight` | A small fixed pool (MECH = 4, ELEC = 2, IC = 1) that does not grow with n. As problem size increases, more activities compete for the same limited slots, forcing heavy resource-driven serialisation. This is the stress-test mode: it exposes worst-case scheduling cost and can trigger the safety cutoff at large n. |
| `plant` | A realistic multi-skill crew pool representative of a nuclear outage staffing plan: MECH = 40, ELEC = 20, IC = 10. The pool is fixed regardless of n, so resource pressure grows naturally with problem size. Unlike `tight`, the capacities are large enough that most plant-scale runs complete; they primarily test scheduler throughput rather than corner-case serialisation. |

---

**Notebook structure**
1. Configuration and benchmark execution
2. Data loading and pre-processing
3. Scheduler benchmark — scaling, exponents, SGS comparison, `_ready` set
4. Plant-scale benchmark — scaling and completion analysis
5. Summary
6. PSPLIB calibration

---

## 1. Setup

In [ ]:
import sys
import json
import math
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)

# The notebook lives in src/CPM/tests/comp_perf/.
# Path('.') is the CWD, which Jupyter sets to the notebook directory.
HERE = Path('.').resolve()   # .../src/CPM/tests/comp_perf
SRC  = HERE.parents[2]      # .../src
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f'Notebook dir : {HERE}')
print(f'LOGOS/src    : {SRC}')

### Configuration

Set `FORCE_RERUN = True` to re-run both benchmark scripts from scratch,
overwriting any existing result files.  Leave it `False` to reload
pre-computed results (much faster).

`SIZES`, `REPS`, and `SGS_VARIANTS` only take effect when `FORCE_RERUN = True`.

In [ ]:
RESULTS_FILE       = HERE / 'benchmark_results.json'
PLANT_RESULTS_FILE = HERE / 'benchmark_plant_results.json'

FORCE_RERUN = False

# --- settings for a fresh run (only used when FORCE_RERUN=True) -----------
SIZES        = [100, 300, 500, 1_000, 1_500]
SIZES_PLANT  = [1_000, 3_000, 5_000, 10_000, 15_000]
REPS         = 3
REPS_PLANT   = 1   # plant runs are slow (10 K–15 K activities); 1 rep is the default
SGS_VARIANTS = ['first', 'max_use_res_ranked', 'max_use_res_shuffled',
                'md_knapsack', 'look_ahead']

# Canonical display order for SGS variants in plots
SGS_ORDER = ['first', 'max_use_res_ranked', 'max_use_res_shuffled',
             'md_knapsack', 'look_ahead']
TOPO_ORDER_SCHED = ['serial', 'fan', 'pipeline']
TOPO_ORDER_PLANT = ['plant_outage', 'pipeline', 'fan', 'serial']


### Running the Benchmarks

The two cells below invoke `benchmark_scheduler.py` and `benchmark_plant.py`
as subprocesses. At default settings (`REPS=3`, all 5 SGS variants, 5 sizes)
the scheduler benchmark takes a few minutes; the plant benchmark with
`REPS=1` and large problem sizes can take 15–30 minutes.

Progress is printed line-by-line as the subprocess runs.

In [ ]:
def _run_benchmark(module: str, out_path: Path, extra_args: list) -> None:
    """Launch a benchmark module as a subprocess and stream its stdout."""
    cmd = [sys.executable, '-m', module, '--out', str(out_path)] + extra_args
    print('Running:', ' '.join(cmd))
    with subprocess.Popen(
        cmd, cwd=str(SRC),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as proc:
        for line in proc.stdout:
            print(line, end='')
    if proc.returncode != 0:
        raise RuntimeError(f'Benchmark exited with code {proc.returncode}')


if FORCE_RERUN or not RESULTS_FILE.exists():
    _run_benchmark(
        'CPM.tests.comp_perf.benchmark_scheduler',
        RESULTS_FILE,
        ['--sizes'] + [str(s) for s in SIZES]
        + ['--sgs']  + SGS_VARIANTS
        + ['--reps', str(REPS)],
    )
else:
    print(f'Skipping scheduler benchmark — will load {RESULTS_FILE}')

In [ ]:
if FORCE_RERUN or not PLANT_RESULTS_FILE.exists():
    _run_benchmark(
        'CPM.tests.comp_perf.benchmark_plant',
        PLANT_RESULTS_FILE,
        ['--sizes'] + [str(s) for s in SIZES_PLANT]
        + ['--sgs']  + SGS_VARIANTS
        + ['--reps', str(REPS_PLANT)],
    )
else:
    print(f'Skipping plant benchmark — will load {PLANT_RESULTS_FILE}')

## 2. Data Loading and Pre-processing

Each JSON result file has two top-level keys:
- `meta` — benchmark parameters (run timestamp, sizes, SGS variants, etc.)
- `results` — list of result dicts, one per `(topology, pool, SGS, n)` combination

Key columns in the DataFrame:

| Column | Description |
|---|---|
| `topology` | Graph structure (`serial`, `fan`, `pipeline`, `plant_outage`) |
| `pool` | Resource pool (`unconstrained`, `tight`, `plant`) |
| `sgs` | SGS variant name |
| `n_work` | Number of work activities |
| `n_total` | Total nodes including gates / START / END |
| `t_cpm_ms` | CPM (`generateInfo`) wall-clock time, ms |
| `t_sched_ms` | Minimum scheduling wall-clock time over reps, ms |
| `iterations` | Scheduler event-loop iterations |
| `n_completed` | Activities successfully scheduled |
| `is_complete` | `True` when `n_completed == n_total` (safety cutoff not hit) |
| `is_valid` | `True` when `validate_schedule()` finds no constraint violations |
| `n_violations` | Number of constraint violations reported by `validate_schedule()` (`-1` if validation itself raised an exception) |
| `peak_ready` | Maximum `_ready` set size across all scheduling steps |
| `avg_ready` | Mean `_ready` set size across all scheduling steps |

Rows where the benchmark raised an exception carry an `error` field and are
dropped.  Incomplete runs (`is_complete = False`, safety cutoff triggered) are
retained but flagged in the analysis.  Invalid runs (`is_valid = False`)
indicate a real correctness failure and are highlighted throughout.

In [ ]:
def load_results(path: Path) -> tuple:
    with open(path) as f:
        raw = json.load(f)
    meta = raw['meta']
    df   = pd.DataFrame(raw['results'])

    # Drop runs that raised an exception
    if 'error' in df.columns:
        bad = df['error'].notna()
        if bad.any():
            print(f'  WARNING: dropped {bad.sum()} error rows from {path.name}')
        df = df[~bad].copy()

    # Back-compat: old result files may lack these columns
    if 'sgs' not in df.columns:
        df['sgs'] = 'max_use_res_ranked'
    if 'is_complete' not in df.columns:
        df['is_complete'] = df['n_completed'] == df['n_total']
    if 'is_valid' not in df.columns:
        df['is_valid'] = True
    if 'n_violations' not in df.columns:
        df['n_violations'] = 0
    if 't_median_ms' not in df.columns:
        df['t_median_ms'] = df['t_sched_ms']   # old files only have min
    if 't_p95_ms' not in df.columns:
        df['t_p95_ms'] = df['t_sched_ms']
    if 'cpm_duration' not in df.columns:
        df['cpm_duration'] = None
    if 'sched_ratio' not in df.columns:
        df['sched_ratio'] = None
    if 'peak_mem_mb' not in df.columns:
        df['peak_mem_mb'] = None

    df = df.dropna(subset=['t_sched_ms']).reset_index(drop=True)
    return meta, df


print('=== Scheduler benchmark ===')
sched_meta, df_sched = load_results(RESULTS_FILE)
print(f'  Run at    : {sched_meta["run_at"]}')
print(f'  Sizes     : {sched_meta["sizes"]}')
print(f'  SGS       : {sched_meta.get("sgs", ["max_use_res_ranked"])}')
print(f'  Rows      : {len(df_sched)}')
incomplete = (~df_sched['is_complete']).sum()
print(f'  Incomplete: {incomplete}')
invalid_s = (~df_sched['is_valid']).sum()
print(f'  Invalid   : {invalid_s}')

print()
print('=== Plant benchmark ===')
plant_meta, df_plant = load_results(PLANT_RESULTS_FILE)
print(f'  Run at    : {plant_meta["run_at"]}')
print(f'  Sizes     : {plant_meta["sizes"]}')
print(f'  SGS       : {plant_meta.get("sgs", ["max_use_res_ranked"])}')
print(f'  Rows      : {len(df_plant)}')
incomplete_p = (~df_plant['is_complete']).sum()
print(f'  Incomplete: {incomplete_p}')
invalid_p = (~df_plant['is_valid']).sum()
print(f'  Invalid   : {invalid_p}')

df_sched.head(3)

### Schedule Validity Check

After each scheduler run, `validate_schedule()` checks all constraint types:
precedence (including lags), crew capacity, equipment capacity, location concurrency,
hold points, and time windows.  `is_valid = False` with `n_violations > 0` signals
a real correctness failure in the SGS implementation.

`n_violations = -1` means validation itself raised an exception (e.g. no schedule
was produced because the safety cutoff was hit).

In [ ]:
# Validation summary across both benchmarks
for label, df in [('Scheduler', df_sched), ('Plant', df_plant)]:
    invalid = df[~df['is_valid']]
    print(f'=== {label} ===')
    if invalid.empty:
        print('  All schedules passed validation. ✓')
    else:
        print(f'  {len(invalid)} INVALID run(s):')
        cols = ['topology', 'pool', 'sgs', 'n_work', 'n_violations', 'is_complete']
        display(invalid[cols].reset_index(drop=True))
    print()

## 3. Scheduler Benchmark Analysis

The following sections cover:
- **Scheduling time scaling** — how wall-clock time grows with problem size
- **CPM vs scheduling overhead** — the minimum possible time vs actual cost
- **Empirical scaling exponents** — quantitative O(n) vs O(n²) characterisation
- **SGS variant comparison** — relative speed of each scheduling strategy
- **Scheduler iterations** — event-loop work as a proxy for algorithmic cost
- **`_ready` set size** — direct driver of candidate-selection cost

---

### Scheduling Time Scaling (log–log)

Wall-clock scheduling time is plotted against problem size on a **log–log** scale.
A straight line indicates a power-law relationship; its slope is the empirical
scaling exponent:

- Slope ≈ **1.0** → O(n) — time grows proportionally with the number of activities ✓
- Slope ≈ **2.0** → O(n²) — quadratic growth, will not scale to 15 K activities ✗

**Open markers / dotted lines** denote incomplete runs (safety cutoff triggered);
their times are artificially low and should not be used for scaling analysis.

In [ ]:
df_plot = df_sched.copy()
df_plot = df_plot.sort_values('n_work')

# Keep only topologies and pools present in the data
topos_present = [t for t in TOPO_ORDER_SCHED if t in df_plot['topology'].unique()]
pools_present = [p for p in ['unconstrained', 'tight'] if p in df_plot['pool'].unique()]

fig = px.line(
    df_plot,
    x='n_work', y='t_sched_ms',
    color='sgs',
    facet_col='topology', facet_row='pool',
    log_x=True, log_y=True,
    markers=True,
    custom_data=['is_complete'],   # embedded per-point so the loop below uses
                                   # per-trace completeness, not a global n set
    title='Scheduling time vs problem size \u2014 log\u2013log (facets: topology \u00d7 pool)<br>'
          '<sup>Open markers = safety-cutoff hit (schedule may be partial)</sup>',
    labels={'n_work': 'n (work activities)', 't_sched_ms': 'Sched time (ms)', 'sgs': 'SGS'},
    category_orders={
        'topology': topos_present,
        'pool':     pools_present,
        'sgs':      SGS_ORDER,
    },
    height=500,
)
# Per-point open-circle for cutoff hits. customdata[i][0] is is_complete for
# point i in this trace; this avoids incorrectly marking complete traces in
# other facets just because a different config hit the cutoff at the same n.
for trace in fig.data:
    if trace.customdata is None:
        continue
    symbols = ['circle-open' if not cd[0] else 'circle' for cd in trace.customdata]
    sizes   = [10           if not cd[0] else 6          for cd in trace.customdata]
    trace.marker.symbol = symbols
    trace.marker.size   = sizes
fig.update_layout(legend_title_text='SGS variant')
fig.show()


### CPM Time vs Scheduling Time

`generateInfo()` runs the classical Critical Path Method (CPM) — a single
forward/backward pass through the activity graph — and is O(V + E) in the
number of nodes and edges.  It represents the **theoretical minimum overhead**
for any scheduling operation.

This plot overlays the CPM time against the **best-case** scheduling time
(minimum over all complete SGS variants) to quantify the additional cost
introduced by resource-constrained scheduling.  When the two lines converge,
the scheduler is close to CPM-limited (graph traversal dominates over
resource-feasibility checks).

In [ ]:
# Best (minimum) scheduling time per (topology, pool, n_work), complete runs only
best = (
    df_sched[df_sched['is_complete']]
    .groupby(['topology', 'pool', 'n_work'], as_index=False)
    .agg(best_sched_ms=('t_sched_ms', 'min'), t_cpm_ms=('t_cpm_ms', 'first'))
)

# Reshape to long form for a unified colour legend
cpm_rows  = best[['topology', 'pool', 'n_work', 't_cpm_ms']].rename(columns={'t_cpm_ms': 'time_ms'}).assign(metric='CPM (generateInfo)')
best_rows = best[['topology', 'pool', 'n_work', 'best_sched_ms']].rename(columns={'best_sched_ms': 'time_ms'}).assign(metric='Best SGS')
long = pd.concat([cpm_rows, best_rows], ignore_index=True).sort_values('n_work')

fig = px.line(
    long,
    x='n_work', y='time_ms',
    color='metric', line_dash='pool',
    facet_col='topology',
    log_x=True, log_y=True,
    markers=True,
    title='CPM time vs best scheduling time (solid=unconstrained, dashed=tight)',
    labels={'n_work': 'n (work activities)', 'time_ms': 'Time (ms)', 'metric': 'Metric'},
    category_orders={'topology': topos_present},
    height=380,
)
fig.show()

### Empirical Scaling Exponents

For each pair of consecutive problem sizes the **empirical exponent** is:

$$e = \frac{\log(t_{\text{new}} / t_{\text{old}})}{\log(n_{\text{new}} / n_{\text{old}})}$$

- $e \approx 1.0$ → O(n) scaling — ideal
- $e \approx 2.0$ → O(n²) — quadratic, will not scale
- $e < 0$ → non-monotone (JIT warm-up, cache effects at small n)

Incomplete runs are **excluded** from exponent computation because their
scheduling times are truncated by the safety cutoff and do not reflect
true algorithmic cost.

The table below shows the mean, minimum, and maximum exponent per
`(topology, pool, SGS)` triple.  The heatmap following it shows the
exponent at each consecutive size step.

In [ ]:
def compute_exponents(df: pd.DataFrame) -> pd.DataFrame:
    """Compute empirical scaling exponents between consecutive size steps."""
    records = []
    for keys, grp in df.groupby(['topology', 'pool', 'sgs']):
        topo, pool, sgs = keys
        rows = grp[grp['is_complete']].sort_values('n_work').reset_index(drop=True)
        for i in range(1, len(rows)):
            prev, curr = rows.iloc[i - 1], rows.iloc[i]
            if prev['t_sched_ms'] <= 0:
                continue
            t_ratio = curr['t_sched_ms'] / prev['t_sched_ms']
            n_ratio = curr['n_work'] / prev['n_work']
            exp = math.log(t_ratio) / math.log(n_ratio) if n_ratio > 1 else float('nan')
            records.append({
                'topology': topo, 'pool': pool, 'sgs': sgs,
                'step': f'{int(prev["n_work"])} -> {int(curr["n_work"])}',
                't_ratio': round(t_ratio, 3),
                'exponent': round(exp, 2),
                'non_monotone': t_ratio < 1.0,
            })
    return pd.DataFrame(records)


exp_df = compute_exponents(df_sched)

# Summary table: mean / min / max exponent per (topology, pool, sgs)
exp_summary = (
    exp_df.groupby(['topology', 'pool', 'sgs'])['exponent']
    .agg(['mean', 'min', 'max'])
    .round(2)
    .rename(columns={'mean': 'mean_exp', 'min': 'min_exp', 'max': 'max_exp'})
    .reset_index()
    .sort_values(['topology', 'pool', 'mean_exp'])
)

# Highlight non-monotone rows
n_nonmono = exp_df['non_monotone'].sum()
if n_nonmono > 0:
    print(f'Note: {n_nonmono} non-monotone step(s) detected (t_ratio < 1) '
          f'— likely JIT warm-up at small n.  Exponent will be negative.')
    print()

display(exp_summary)

In [ ]:
if exp_df.empty:
    print('No consecutive size pairs with complete runs \u2014 heatmap unavailable.')
else:
    # Heatmap: rows = (topology / pool / sgs),  columns = size step
    pivot = exp_df.pivot_table(
        index=['topology', 'pool', 'sgs'],
        columns='step',
        values='exponent',
        aggfunc='first',
    )
    # Flatten multi-index row labels for display
    pivot.index = [' / '.join(idx) for idx in pivot.index]

    n_rows = max(400, 120 + len(pivot) * 40)  # 40 px/row + 120 px base for title/axes

    fig = px.imshow(
        pivot,
        color_continuous_scale='RdYlGn_r',   # green = low exponent (fast), red = high (slow)
        zmin=0.0, zmax=2.5,
        text_auto='.2f',
        aspect='auto',
        title='Empirical scaling exponent per size step '
              '(green \u2248 O(n), yellow \u2248 O(n\u00b9\u22c5\u2075), red \u2248 O(n\u00b2))',
        labels={'x': 'Size step  (n_old -> n_new)', 'y': 'Config', 'color': 'Exponent'},
        height=n_rows,
    )
    fig.update_layout(xaxis_tickangle=-25)
    fig.show()


### SGS Variant Comparison at Maximum n

This bar chart compares scheduling wall-clock time across all five SGS variants
at the **largest complete n for each topology/pool combination**, separately for
each facet.  Configurations where the safety cutoff was hit at the global maximum
n fall back to their own largest complete n, so all pool rows remain visible.

Only complete runs are shown.  A faster SGS does not necessarily produce a
better (shorter) outage schedule — it may make greedier or less thorough
decisions.  This chart measures **computational cost**, not **solution quality**.

In [ ]:
# Per-(topology, pool) best complete n \u2014 avoids silently dropping pools where
# the global max n has no complete runs.
_rows = []
for (topo, pool), grp in df_sched.groupby(['topology', 'pool']):
    _complete = grp[grp['is_complete']]
    if _complete.empty:
        continue
    _best_n = _complete['n_work'].max()
    _rows.append(_complete[_complete['n_work'] == _best_n])
df_max = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()

if df_max.empty:
    print('No complete scheduler runs available for SGS comparison.')
else:
    # _n_label must be built inside the else: df_max has no columns when empty
    _n_label = (f"n={df_max['n_work'].iloc[0]}" if df_max['n_work'].nunique() == 1
                else f"n={df_max['n_work'].min()}\u2013{df_max['n_work'].max()}")
    fig = px.bar(
        df_max.sort_values('t_sched_ms'),
        x='sgs', y='t_sched_ms',
        color='sgs',
        facet_col='topology', facet_row='pool',
        title=f'Scheduling time by SGS variant at max complete n per config ({_n_label})',
        labels={'sgs': 'SGS', 't_sched_ms': 'Sched time (ms)'},
        category_orders={'topology': topos_present, 'pool': pools_present, 'sgs': SGS_ORDER},
        height=500,
        text_auto='.1f',
    )
    fig.update_layout(showlegend=False)
    fig.update_xaxes(tickangle=35)
    fig.show()


### Schedule Quality Ratio (Sched duration / CPM duration)

A ratio of **1.0** means the resource-constrained schedule achieves the same
makespan as the unconstrained CPM critical path — no resource-induced delay.
Higher values indicate resource contention stretching the project.  This lets
you distinguish a *fast but poor* SGS from a *fast and good* one.

Only complete runs are shown; incomplete runs (safety cutoff hit) are excluded
because their scheduled duration is a lower bound, not a true makespan.

In [ ]:
df_ratio = df_sched[df_sched['is_complete'] & df_sched['sched_ratio'].notna()].copy()

if df_ratio.empty:
    print('No sched_ratio data available (result files predate this metric).')
else:
    fig = px.line(
        df_ratio.sort_values('n_work'),
        x='n_work', y='sched_ratio',
        color='sgs',
        facet_col='topology', facet_row='pool',
        markers=True,
        log_x=True,
        title='Schedule quality ratio (sched_duration / cpm_duration) — closer to 1.0 is better',
        labels={'n_work': 'n (work activities)', 'sched_ratio': 'Sched / CPM', 'sgs': 'SGS'},
        category_orders={
            'topology': TOPO_ORDER_SCHED,
            'pool':     ['unconstrained', 'tight'],
            'sgs':      SGS_ORDER,
        },
        height=500,
    )
    fig.add_hline(y=1.0, line_dash='dash', line_color='green',
                  annotation_text='CPM lower bound')
    fig.update_layout(legend_title_text='SGS variant')
    fig.show()

### Scheduler Iterations

The scheduler's event loop advances time from one event (activity completion
or resource availability change) to the next.  The **iteration count** is a
topology- and pool-dependent proxy for algorithmic cost that is independent
of machine speed.

- **Serial**: one iteration per activity + START/END — strictly O(n)
- **Pipeline (unconstrained)**: iterations ≈ `n_clusters × 3` (gate-in, work, gate-out events)
- **Fan (tight)**: iterations grow with `n / parallel_slots` — more serialisation = more steps

Since the iteration count is determined by the graph structure and resource
pressure (not the SGS selection logic), all SGS variants should show very
similar iteration counts for the same `(topology, pool, n)`.

In [ ]:
# Average iterations across SGS variants (they should be nearly identical)
iters = (
    df_sched[df_sched['is_complete']]
    .groupby(['topology', 'pool', 'n_work'], as_index=False)
    .agg(iterations=('iterations', 'mean'), std_iter=('iterations', 'std'))
)

fig = px.line(
    iters.sort_values('n_work'),
    x='n_work', y='iterations',
    color='topology', line_dash='pool',
    markers=True,
    log_x=True, log_y=True,
    title='Scheduler iterations vs n (solid=unconstrained, dashed=tight)',
    labels={'n_work': 'n (work activities)', 'iterations': 'Event-loop iterations'},
    category_orders={'topology': topos_present},
    height=400,
)
fig.show()

# std_iter is NaN when only one SGS variant was benchmarked (std of a single value).
# A non-zero std would signal that iterations vary across SGS \u2014 which should not happen.
print('Max std-dev in iterations across SGS variants:')
valid_std = iters[iters['std_iter'].notna()]
if valid_std.empty:
    print('  (all NaN \u2014 only one SGS variant in the data; no cross-SGS comparison possible)')
else:
    display(valid_std.nlargest(5, 'std_iter')[['topology', 'pool', 'n_work', 'iterations', 'std_iter']])


### `_ready` Set Size

At each scheduling step the `_ready` set contains all activities whose
predecessors are complete.  Its size directly controls the cost of the
candidate-selection inner loop — the most expensive part of the scheduler.

- **Serial**: `_ready` is always 1 — the inner loop is trivial
- **Pipeline**: `_ready` ≤ `cluster_size` (10) — bounded regardless of n
- **Fan**: `_ready` starts at n — worst case for candidate-selection cost

Because `_ready` is a property of the graph topology and resource pressure
(not of which SGS is used), the values are averaged across SGS variants.
The `peak_ready` column reflects the maximum seen in a single run;
`avg_ready` is the mean over all `_select_candidate_activities` calls.

In [ ]:
# Average _ready statistics across SGS variants
ready = (
    df_sched
    .groupby(['topology', 'pool', 'n_work'], as_index=False)
    .agg(peak_ready=('peak_ready', 'mean'), avg_ready=('avg_ready', 'mean'))
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Peak _ready set size', 'Avg _ready set size'],
    shared_xaxes=False,
)

palette = px.colors.qualitative.Plotly
topo_color = {t: palette[i] for i, t in enumerate(TOPO_ORDER_SCHED)}
pool_dash  = {'unconstrained': 'solid', 'tight': 'dash'}

seen = set()
for _, grp in ready.groupby(['topology', 'pool']):
    topo = grp['topology'].iloc[0]
    pool = grp['pool'].iloc[0]
    label = f'{topo} / {pool}'
    grp = grp.sort_values('n_work')
    show = label not in seen
    seen.add(label)
    common = dict(
        x=grp['n_work'], mode='lines+markers', name=label,
        legendgroup=label,
        line=dict(color=topo_color.get(topo, 'grey'),
                  dash=pool_dash.get(pool, 'solid')),
    )
    fig.add_trace(go.Scatter(y=grp['peak_ready'], showlegend=show,  **common), row=1, col=1)
    fig.add_trace(go.Scatter(y=grp['avg_ready'],  showlegend=False, **common), row=1, col=2)
fig.update_xaxes(title_text='n (work activities)')
fig.update_yaxes(title_text='Set size', col=1)
fig.update_layout(
    title='_ready set size vs problem size (solid=unconstrained, dashed=tight)',
    height=420, legend_title='topology / pool',
)
fig.show()

## 4. Plant-Scale Benchmark Analysis

The plant-scale benchmark uses realistic nuclear outage conditions:
- **`plant_outage` topology**: 16 parallel work-package pipelines with
  15 % of streams gated on the completion of the preceding stream
  (system-isolation prerequisites)
- **Randomised durations**: drawn from a plant-realistic distribution
  `[1h, 2h, 4h, 8h, 16h, 24h]` with weights `[15, 20, 30, 20, 10, 5]`
- **Multi-skill resource pool** (`plant` mode): MECH=40 / ELEC=20 / IC=10,
  fixed regardless of n — constraint pressure grows naturally with problem size
- **Problem sizes**: 1 000 – 15 000 work activities

---

### Plant Scheduling Time Scaling

In [ ]:
topos_plant   = [t for t in TOPO_ORDER_PLANT if t in df_plant['topology'].unique()]
pools_plant   = [p for p in ['plant', 'tight', 'unconstrained'] if p in df_plant['pool'].unique()]

df_pplot = df_plant.copy()
df_pplot = df_pplot.sort_values('n_work')

fig = px.line(
    df_pplot,
    x='n_work', y='t_sched_ms',
    color='sgs',
    facet_col='topology', facet_row='pool',
    log_x=True, log_y=True,
    markers=True,
    custom_data=['is_complete'],   # embedded per-point so the loop below uses
                                   # per-trace completeness, not a global n set
    title='Plant-scale scheduling time vs n \u2014 log\u2013log (facets: topology \u00d7 pool)<br>'
          '<sup>Open markers = safety-cutoff hit (schedule may be partial)</sup>',
    labels={'n_work': 'n (work activities)', 't_sched_ms': 'Sched time (ms)', 'sgs': 'SGS'},
    category_orders={
        'topology': topos_plant,
        'pool':     pools_plant,
        'sgs':      SGS_ORDER,
    },
    height=600,
)
# Per-point open-circle for cutoff hits. customdata[i][0] is is_complete for
# point i in this trace; this avoids incorrectly marking complete traces in
# other facets just because a different config hit the cutoff at the same n.
for trace in fig.data:
    if trace.customdata is None:
        continue
    symbols = ['circle-open' if not cd[0] else 'circle' for cd in trace.customdata]
    sizes   = [10           if not cd[0] else 6          for cd in trace.customdata]
    trace.marker.symbol = symbols
    trace.marker.size   = sizes
fig.update_layout(legend_title_text='SGS variant')
fig.show()


### Plant Empirical Scaling Exponents

Same exponent analysis applied to the plant-scale results.  The fixed crew
pool (`plant` mode: MECH=40, ELEC=20, IC=10) means constraint pressure
increases with n, unlike the `unconstrained` pool.
Exponents above 1.5 at outage scale are a warning sign that a particular
SGS + topology combination will not handle 15 K activities in reasonable
wall-clock time.

In [ ]:
exp_plant = compute_exponents(df_plant)

exp_plant_summary = (
    exp_plant.groupby(['topology', 'pool', 'sgs'])['exponent']
    .agg(['mean', 'min', 'max'])
    .round(2)
    .rename(columns={'mean': 'mean_exp', 'min': 'min_exp', 'max': 'max_exp'})
    .reset_index()
    .sort_values(['topology', 'pool', 'mean_exp'])
)

display(exp_plant_summary)

if not exp_plant.empty:
    pivot_p = exp_plant.pivot_table(
        index=['topology', 'pool', 'sgs'],
        columns='step',
        values='exponent',
        aggfunc='first',
    )
    pivot_p.index = [' / '.join(idx) for idx in pivot_p.index]
    fig = px.imshow(
        pivot_p,
        color_continuous_scale='RdYlGn_r',
        zmin=0.0, zmax=2.5,
        text_auto='.2f',
        aspect='auto',
        title='Plant-scale empirical exponents (green ≈ O(n), red ≈ O(n²))',
        labels={'x': 'Size step', 'y': 'Config', 'color': 'Exponent'},
        height=max(400, 120 + len(pivot_p) * 40),
    )
    fig.update_layout(xaxis_tickangle=-25)
    fig.show()

### Completion Rate Analysis

When the scheduler's safety cutoff (`max_time_hours`) is reached before all
activities are scheduled, the run is marked `is_complete = False`.  These runs
indicate that the configured time horizon was insufficient — either the resource
constraints force a much longer schedule, or the SGS is getting stuck.

The chart below shows the fraction of activities completed (`n_completed / n_total`)
for each configuration.  A value < 1.0 signals an incomplete run.

In [ ]:
df_plant['completion_rate'] = df_plant['n_completed'] / df_plant['n_total']

fig = px.line(
    df_plant.sort_values('n_work'),
    x='n_work', y='completion_rate',
    color='sgs',
    facet_col='topology', facet_row='pool',
    markers=True,
    range_y=[0.0, 1.05],
    title='Completion rate (n_completed / n_total) — values < 1.0 indicate cutoff',
    labels={'n_work': 'n (work activities)', 'completion_rate': 'Completion rate', 'sgs': 'SGS'},
    category_orders={'topology': topos_plant, 'pool': pools_plant, 'sgs': SGS_ORDER},
    height=600,
)
fig.add_hline(y=1.0, line_dash='dash', line_color='green', annotation_text='100 % complete')
fig.update_layout(legend_title_text='SGS variant')
fig.show()

# Table of incomplete runs
incomplete_plant = df_plant[~df_plant['is_complete']][
    ['topology', 'pool', 'sgs', 'n_work', 'n_completed', 'n_total', 'completion_rate', 't_sched_ms']
].sort_values(['topology', 'pool', 'n_work'])

if incomplete_plant.empty:
    print('No incomplete runs in plant benchmark.')
else:
    print(f'{len(incomplete_plant)} incomplete run(s):')
    display(incomplete_plant)

### SGS Comparison at Maximum Plant n

Same as the scheduler SGS bar chart but at plant scale.  Each facet shows the
**largest complete n for that topology/pool combination** — if the global
maximum n has no complete runs for a given pool (safety cutoff hit for all SGS
variants), that facet falls back to its own best complete n rather than being
omitted.  Configurations where no complete run exists at any n are excluded.

In [ ]:
# Per-(topology, pool) best complete n \u2014 avoids silently dropping pools where
# the global max n has no complete runs.
_rows_p = []
for (topo, pool), grp in df_plant.groupby(['topology', 'pool']):
    _complete = grp[grp['is_complete']]
    if _complete.empty:
        continue
    _best_n = _complete['n_work'].max()
    _rows_p.append(_complete[_complete['n_work'] == _best_n])
df_pmax = pd.concat(_rows_p, ignore_index=True) if _rows_p else pd.DataFrame()

if df_pmax.empty:
    print('No complete plant runs available for SGS comparison.')
else:
    # _n_label_p must be built inside the else: df_pmax has no columns when empty
    _n_label_p = (f"n={df_pmax['n_work'].iloc[0]}" if df_pmax['n_work'].nunique() == 1
                  else f"n={df_pmax['n_work'].min()}\u2013{df_pmax['n_work'].max()}")
    fig = px.bar(
        df_pmax.sort_values('t_sched_ms'),
        x='sgs', y='t_sched_ms',
        color='sgs',
        facet_col='topology', facet_row='pool',
        title=f'Plant SGS comparison at max complete n per config ({_n_label_p})',
        labels={'sgs': 'SGS', 't_sched_ms': 'Sched time (ms)'},
        category_orders={'topology': topos_plant, 'pool': pools_plant, 'sgs': SGS_ORDER},
        height=550,
        text_auto='.0f',
    )
    fig.update_layout(showlegend=False)
    fig.update_xaxes(tickangle=35)
    fig.show()


### Plant Schedule Quality Ratio

Same quality metric as the scheduler benchmark — `sched_duration / cpm_duration`
for plant-scale configurations.

In [ ]:
df_pratio = df_plant[df_plant['is_complete'] & df_plant['sched_ratio'].notna()].copy()

if df_pratio.empty:
    print('No sched_ratio data available (result files predate this metric).')
else:
    fig = px.line(
        df_pratio.sort_values('n_work'),
        x='n_work', y='sched_ratio',
        color='sgs',
        facet_col='topology', facet_row='pool',
        markers=True,
        log_x=True,
        title='Plant schedule quality ratio (sched_duration / cpm_duration)',
        labels={'n_work': 'n (work activities)', 'sched_ratio': 'Sched / CPM', 'sgs': 'SGS'},
        category_orders={
            'topology': TOPO_ORDER_PLANT,
            'pool':     ['plant', 'tight', 'unconstrained'],
            'sgs':      SGS_ORDER,
        },
        height=500,
    )
    fig.add_hline(y=1.0, line_dash='dash', line_color='green',
                  annotation_text='CPM lower bound')
    fig.update_layout(legend_title_text='SGS variant')
    fig.show()

## 5. Summary

The table below consolidates the key performance indicators across both
benchmarks: the **mean scaling exponent** (lower is better), the
**maximum scheduling time** at the largest complete n, and a
**completion flag** indicating whether the safety cutoff was hit at any size.

Use this table to identify which `(topology, pool, SGS)` combinations
are viable candidates for production use at outage scale.

In [ ]:
def build_summary(df: pd.DataFrame, exp_df_in: pd.DataFrame, label: str) -> pd.DataFrame:
    # Mean exponent per config
    exp_agg = (
        exp_df_in.groupby(['topology', 'pool', 'sgs'])['exponent']
        .mean().round(2).rename('mean_exp').reset_index()
    )
    # Max time at largest complete n per config
    _complete = df[df['is_complete']]
    if _complete.empty:
        max_time = pd.DataFrame(columns=['topology', 'pool', 'sgs', 'max_t_sched_ms'])
    else:
        _idx = _complete.groupby(['topology', 'pool', 'sgs'])['n_work'].idxmax()
        max_time = (
            _complete.loc[_idx, ['topology', 'pool', 'sgs', 't_sched_ms']]
            .rename(columns={'t_sched_ms': 'max_t_sched_ms'})
            .assign(max_t_sched_ms=lambda x: x['max_t_sched_ms'].round(1))
            .reset_index(drop=True)
        )
    # Completion flag
    comp = (
        df.groupby(['topology', 'pool', 'sgs'])['is_complete']
        .all().rename('all_complete').reset_index()
    )
    summary = exp_agg.merge(max_time, on=['topology', 'pool', 'sgs'], how='left')
    summary = summary.merge(comp,     on=['topology', 'pool', 'sgs'], how='left')
    summary.insert(0, 'benchmark', label)
    return summary.sort_values(['topology', 'pool', 'mean_exp'])


sched_summary = build_summary(df_sched,  exp_df,    'scheduler')
plant_summary = build_summary(df_plant,  exp_plant, 'plant')

combined = pd.concat([sched_summary, plant_summary], ignore_index=True)
combined['all_complete'] = combined['all_complete'].map({True: 'yes', False: 'NO ⚠'})
combined = combined.rename(columns={
    'mean_exp':        'mean scaling exp',
    'max_t_sched_ms':  'max time (ms) @ largest n',
    'all_complete':    'all runs complete',
})

display(combined)


In [ ]:
# Highlight the fastest complete SGS per (benchmark, topology, pool)
fastest = (
    combined[combined['all runs complete'] == 'yes']
    .sort_values('max time (ms) @ largest n')
    .groupby(['benchmark', 'topology', 'pool'], as_index=False)
    .first()
    [['benchmark', 'topology', 'pool', 'sgs', 'mean scaling exp', 'max time (ms) @ largest n']]
)

print('Fastest complete SGS per (benchmark, topology, pool):')
display(fastest)

## 6. PSPLIB Calibration

This section loads results from `benchmark_psplib.py` and compares each SGS
variant's makespan against the **known best solution** from the PSPLIB database.

- **Gap (%)** = `(sched_duration − optimal) / optimal × 100`.  A gap of 0 % means
  the SGS recovered the optimal (or best-known) makespan.
- **Sched/CPM** = schedule quality ratio (same as other sections).
- **Peak mem MB** = peak heap allocation during `calculateScheduleWithResources`
  measured with `tracemalloc`.

Instances are pre-converted from PSPLIB `.sm` format to LOGOS outage JSON via
`src/CPM/psplib_converter.ipynb`.  Add more instances by converting additional
`.sm` files and registering them in `_INSTANCE_REGISTRY` inside `benchmark_psplib.py`.

In [ ]:
PSPLIB_RESULTS_FILE = HERE / 'benchmark_psplib_results.json'

if not PSPLIB_RESULTS_FILE.exists():
    print('PSPLIB results not found — run benchmark_psplib.py first:')
    print('  cd .../LOGOS/src')
    print('  python -m CPM.tests.comp_perf.benchmark_psplib')
else:
    with open(PSPLIB_RESULTS_FILE) as _f:
        _raw = json.load(_f)
    df_psp = pd.DataFrame(_raw['results'])
    if 'error' in df_psp.columns:
        df_psp = df_psp[df_psp['error'].isna()].copy()

    # ── Optimality gap bar chart ───────────────────────────────────────────
    fig = px.bar(
        df_psp.sort_values('gap_pct'),
        x='sgs', y='gap_pct',
        color='sgs', facet_col='instance',
        text_auto='.1f',
        title='PSPLIB optimality gap by SGS variant (lower is better; 0 % = optimal)',
        labels={'gap_pct': 'Gap vs optimal (%)', 'sgs': 'SGS variant'},
        category_orders={'sgs': SGS_ORDER},
        height=420,
    )
    fig.add_hline(y=0, line_dash='dash', line_color='green',
                  annotation_text='Optimal')
    fig.update_layout(showlegend=False)
    fig.update_xaxes(tickangle=35)
    fig.show()

    # ── Timing bar chart ──────────────────────────────────────────────────
    fig2 = px.bar(
        df_psp.sort_values('t_sched_ms'),
        x='sgs', y='t_sched_ms',
        color='sgs', facet_col='instance',
        text_auto='.1f',
        title='PSPLIB scheduling time by SGS variant (min over reps)',
        labels={'t_sched_ms': 'Min time (ms)', 'sgs': 'SGS variant'},
        category_orders={'sgs': SGS_ORDER},
        height=420,
    )
    fig2.update_layout(showlegend=False)
    fig2.update_xaxes(tickangle=35)
    fig2.show()

    # ── Summary table ─────────────────────────────────────────────────────
    cols = ['instance', 'sgs', 'optimal_makespan', 'cpm_duration',
            'sched_duration', 'gap_pct', 'sched_ratio',
            't_sched_ms', 't_median_ms', 't_p95_ms', 'peak_mem_mb',
            'is_valid', 'n_violations']
    display(df_psp[[c for c in cols if c in df_psp.columns]].reset_index(drop=True))